<a href="https://colab.research.google.com/github/NVHau-K14/Tuan03_ThucHanh_DeepLearning/blob/main/ANN_DuBaoThuNhap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Đặt seed cho reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

In [ ]:
# Tên cột của bộ dữ liệu Adult
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

# Tải dữ liệu trực tiếp từ UCI Repository
url_train = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
url_test  = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test'

try:
    df_train = pd.read_csv(url_train, names=column_names,
                           sep=',\s*', engine='python', na_values='?')
    df_test  = pd.read_csv(url_test,  names=column_names,
                           sep=',\s*', engine='python', na_values='?', skiprows=1)
    # Gộp train + test để xử lý thống nhất
    df = pd.concat([df_train, df_test], ignore_index=True)
    print(f"Tải dữ liệu thành công từ UCI Repository")
except Exception as e:
    # Fallback: thử tải từ nguồn khác
    print(f"Không tải được từ UCI, thử nguồn dự phòng... ({e})")
    alt_url = 'https://raw.githubusercontent.com/dsrscientist/dataset1/master/adult.csv'
    df = pd.read_csv(alt_url)
    # Chuẩn hoá tên cột
    df.columns = [c.strip().lower().replace('-', '_').replace(' ', '_') for c in df.columns]
    if 'income_bracket' in df.columns:
        df.rename(columns={'income_bracket': 'income'}, inplace=True)
    print("Tải dữ liệu dự phòng thành công")

print(f"\nKích thước bộ dữ liệu: {df.shape}")
df.head()

In [ ]:
print("=" * 60)
print("THÔNG TIN CƠ BẢN")
print("=" * 60)
df.info()
print("\n" + "=" * 60)
print("THỐNG KÊ MÔ TẢ")
print("=" * 60)
df.describe()

In [ ]:
print("\nPhân phối nhãn (income):")
# Chuẩn hoá nhãn (bộ test có dấu chấm cuối)
df['income'] = df['income'].astype(str).str.strip().str.replace('.', '', regex=False)
print(df['income'].value_counts())

print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phân phối nhãn
income_counts = df['income'].value_counts()
colors = ['#4CAF50', '#F44336']
axes[0].pie(income_counts.values, labels=income_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=[0.05, 0])
axes[0].set_title('Phân phối Thu nhập', fontsize=14, fontweight='bold')

# Phân phối tuổi theo nhãn
for label, color in zip(['<=50K', '>50K'], colors):
    subset = df[df['income'] == label]['age']
    axes[1].hist(subset, bins=25, alpha=0.6, label=label, color=color)
axes[1].set_xlabel('Tuổi', fontsize=12)
axes[1].set_ylabel('Tần số', fontsize=12)
axes[1].set_title('Phân phối Tuổi theo Thu nhập', fontsize=14, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# -------- 5.1 Xử lý Missing Values --------
print("Xử lý missing values...")
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols   = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Điền mode cho categorical, median cho numerical
for col in categorical_cols:
    if col != 'income':
        df[col].fillna(df[col].mode()[0], inplace=True)
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

print(f"  Missing sau xử lý: {df.isnull().sum().sum()}")

# -------- 5.2 Loại bỏ cột không cần thiết --------
# 'fnlwgt' là trọng số điều tra - không có ý nghĩa dự báo
if 'fnlwgt' in df.columns:
    df.drop('fnlwgt', axis=1, inplace=True)
    print("  Đã xoá cột 'fnlwgt'")

# -------- 5.3 Encode nhãn mục tiêu --------
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
print(f"\nNhãn sau encode: 0 = <=50K, 1 = >50K")
print(df['income'].value_counts())

In [ ]:
# -------- 5.4 Encode các cột categorical bằng Label Encoding --------
le = LabelEncoder()
cat_features = [c for c in df.select_dtypes(include='object').columns if c != 'income']

for col in cat_features:
    df[col] = le.fit_transform(df[col].astype(str))

print("Label Encoding hoàn tất cho:", cat_features)

# -------- 5.5 Tách features & target --------
X = df.drop('income', axis=1)
y = df['income']

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape  : {y.shape}")
print(f"Số features   : {X.shape[1]}")
print("Tên features  :", list(X.columns))

In [ ]:
# -------- 5.6 Chia Train / Validation / Test --------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train size      : {X_train.shape[0]} mẫu ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Validation size : {X_val.shape[0]} mẫu ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test size       : {X_test.shape[0]} mẫu ({X_test.shape[0]/len(X)*100:.1f}%)")

# -------- 5.7 Chuẩn hoá dữ liệu (StandardScaler) --------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("\nChuẩn hoá dữ liệu xong (StandardScaler)")

In [ ]:
def build_ann(input_dim):
    """
    Xây dựng mạng ANN với kiến trúc:
    Input -> Dense(128) -> BN -> Dropout -> Dense(64) -> BN -> Dropout
          -> Dense(32) -> BN -> Dropout -> Dense(1, sigmoid)
    """
    model = keras.Sequential([
        # --- Lớp đầu vào ---
        layers.Input(shape=(input_dim,)),

        # --- Hidden layer 1 ---
        layers.Dense(128, activation='relu',
                     kernel_initializer='he_normal',
                     kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # --- Hidden layer 2 ---
        layers.Dense(64, activation='relu',
                     kernel_initializer='he_normal',
                     kernel_regularizer=keras.regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # --- Hidden layer 3 ---
        layers.Dense(32, activation='relu',
                     kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        # --- Output layer (binary classification) ---
        layers.Dense(1, activation='sigmoid')
    ], name='ANN_Income_Predictor')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

model = build_ann(X_train_scaled.shape[1])
model.summary()

In [ ]:
# Vẽ kiến trúc mạng bằng text
print("\nKIẾN TRÚC MẠNG ANN")
print("=" * 55)
print(f"  INPUT  : {X_train_scaled.shape[1]} neurons (features)")
print("    ↓")
print("  HIDDEN 1: 128 neurons (ReLU) + BatchNorm + Dropout(0.3)")
print("    ↓")
print("  HIDDEN 2:  64 neurons (ReLU) + BatchNorm + Dropout(0.3)")
print("    ↓")
print("  HIDDEN 3:  32 neurons (ReLU) + BatchNorm + Dropout(0.2)")
print("    ↓")
print("  OUTPUT :    1 neuron  (Sigmoid) → P(income > 50K)")
print("=" * 55)
print(f"  Optimizer : Adam (lr=0.001)")
print(f"  Loss      : Binary Cross-Entropy")
print(f"  Metrics   : Accuracy, AUC")

In [ ]:
# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Tính class weights để xử lý imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(f"Class weights: {class_weight_dict}")

# Huấn luyện
print("\nBắt đầu huấn luyện...")
history = model.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Quá trình huấn luyện ANN', fontsize=16, fontweight='bold')

epochs_ran = range(1, len(history.history['loss']) + 1)

# --- Loss ---
axes[0].plot(epochs_ran, history.history['loss'],     'b-o', markersize=3, label='Train Loss')
axes[0].plot(epochs_ran, history.history['val_loss'], 'r-o', markersize=3, label='Val Loss')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Accuracy ---
axes[1].plot(epochs_ran, history.history['accuracy'],     'b-o', markersize=3, label='Train Acc')
axes[1].plot(epochs_ran, history.history['val_accuracy'], 'r-o', markersize=3, label='Val Acc')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- AUC ---
axes[2].plot(epochs_ran, history.history['auc'],     'b-o', markersize=3, label='Train AUC')
axes[2].plot(epochs_ran, history.history['val_auc'], 'r-o', markersize=3, label='Val AUC')
axes[2].set_title('AUC-ROC', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"  Số epochs thực tế: {len(epochs_ran)}")

In [ ]:
# Dự đoán
y_pred_proba = model.predict(X_test_scaled, verbose=0).flatten()
y_pred       = (y_pred_proba >= 0.5).astype(int)

# Các chỉ số đánh giá
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST")
print("=" * 60)
print(f"  Accuracy  : {acc*100:.2f}%")
print(f"  AUC-ROC   : {auc:.4f}")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
                            target_names=['<=50K', '>50K']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Đánh giá Mô hình ANN', fontsize=16, fontweight='bold')

# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['<=50K', '>50K'],
            yticklabels=['<=50K', '>50K'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('Thực tế')
axes[0].set_xlabel('Dự đoán')

# Thêm % vào từng ô
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / cm[i].sum() * 100
        axes[0].text(j + 0.5, i + 0.7, f'({pct:.1f}%)',
                     ha='center', va='center', fontsize=9, color='gray')

# --- ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, 'b-', linewidth=2,
             label=f'ANN (AUC = {auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()